# SafeShift — HuggingFaceTB/SmolVLM2-2.2B-Instruct CPU environment preflight

Run this notebook on the intended Kaggle image before issuing a new attempt.
It needs Internet for pinned package installation, but no dataset, grant, GPU,
model weights or inference. The Python 3.13 kernel remains the supervisor only.
The preflight prefers `/usr/bin/python3.12`, accepts only qualified Linux x86_64
CPython 3.11/3.12, creates a fresh scratch venv and uses it for the exact pinned
CUDA-wheel install, `pip check`, `pip freeze --all`, version checks and CPU-only
native library smoke. The venv is created `--without-pip --copies`; missing
`ensurepip` is allowed. A source-locked pip 25.0.1 wheel is verified before the
supervisor uses `pip --python` to install it only into the target venv. Missing
`venv`, bootstrap/controller/hash failure, dependency failure, host-package
leakage or wrong versions produce a BLOCKED receipt and consume no attempt.

For Phi this downloads only hash-locked public source/tokenizer files and runs
the real reviewed class `from_pretrained(None, state_dict={})` on full meta
(no checkpoint weights/CUDA/forward), followed by a native synthetic RGB processor
check (16 crops, image tokens, masks, shapes and dtypes). It checks
explicit eager in both Phi and its independent CLIP config, and all 32 decoder
and 24 vision attention classes. Source/config/import/attention failures block
before any attempt. This executes reviewed remote config/constructor code on CPU.

All venv/cache/temp bytes stay under `/kaggle/tmp`; receipts/logs and a ZIP stay
under `/kaggle/working`. Weights are not downloaded and CUDA is not called.
Download the printed ZIP and sidecar. PASS proves only this CPU environment
rehearsal at that time, not model load, T4 fit, inference or HNLO validity.


In [ ]:
import pathlib, json, hashlib, zipfile, uuid
try:
    # Generated by scripts/build_three_vlm_notebooks.py; do not edit.
    import base64, hashlib, json, pathlib, sys, zlib
    payload = json.loads(zlib.decompress(base64.b64decode('...')))
    expected_source_sha256 = 'df1a36ac670e28f511011746265bf5ff19819360744fc9e5e89f08535addd071'
    inventory = {p: hashlib.sha256(t.encode()).hexdigest() for p,t in payload.items()}
    actual = hashlib.sha256((json.dumps(inventory,sort_keys=True,ensure_ascii=True,separators=(",",":"),allow_nan=False)+"\n").encode()).hexdigest()
    if actual != expected_source_sha256:
        raise RuntimeError("EMBEDDED_SOURCE_CORRUPT")
    root = pathlib.Path("/kaggle/working/data/processed/three_vlm_code") / actual
    if any(p.is_symlink() for p in (root, *root.parents)):
        raise RuntimeError("EMBEDDED_ROOT_ALIAS")
    root.mkdir(parents=True, exist_ok=True)
    for name, text in payload.items():
        p = root / name
        if pathlib.PurePosixPath(name).is_absolute() or ".." in pathlib.PurePosixPath(name).parts:
            raise RuntimeError("UNSAFE_EMBEDDED_PATH")
        if any(parent.is_symlink() for parent in (p, *p.parents)):
            raise RuntimeError("EMBEDDED_PATH_ALIAS")
        p.parent.mkdir(parents=True,exist_ok=True)
        if p.exists():
            if p.read_bytes() != text.encode():
                raise RuntimeError("EXISTING_SOURCE_CHANGED")
        else:
            with p.open("xb") as stream:
                stream.write(text.encode())
    sys.path.insert(0,str(root))
    if "safeshift" in sys.modules and not pathlib.Path(sys.modules["safeshift"].__file__).resolve().is_relative_to(root.resolve()):
        raise RuntimeError("STALE_IMPORTED_SAFESHIFT_REQUIRES_CLEAN_SESSION")
    from safeshift.pilot.bootstrap import environment_preflight
except BaseException as error:
    directory = pathlib.Path("/kaggle/working/data/processed/three_vlm_bootstrap") / uuid.uuid4().hex
    directory.mkdir(parents=True, exist_ok=False)
    record = {"status":"BLOCKED", "stage":"notebook_bootstrap", "attempt_consumed":False,
              "type":type(error).__name__, "constraint":str(error)[:300],
              "source_sha256":globals().get("expected_source_sha256"),
              "cuda":"NOT_CALLED", "weights":"NOT_REQUESTED"}
    payload_bytes = (json.dumps(record, sort_keys=True)+"\n").encode()
    (directory / "failure.json").write_bytes(payload_bytes)
    hashes = {"failure.json":hashlib.sha256(payload_bytes).hexdigest()}
    (directory / "checksums.json").write_text(json.dumps(hashes), encoding="utf-8")
    archive = directory.with_suffix(".zip")
    with zipfile.ZipFile(archive,"x",compression=zipfile.ZIP_DEFLATED) as packed:
        for name in ("failure.json","checksums.json"):
            packed.write(directory / name,name)
    (pathlib.Path(str(archive)+".sha256.json")).write_text(json.dumps({"sha256":hashlib.sha256(archive.read_bytes()).hexdigest()}),encoding="utf-8")
    print("BLOCKED before attempt: download", archive, "and its .sha256.json")
    raise
evidence = environment_preflight('smolvlm2', expected_source_sha256, root)
print("Evidence directory:", evidence)
